# Classification of research publications based on data from OpenAlex

Only papers from the CZI that also appear in the research publication classification dataset were included. The original dataset contained 14,770,209 software mentions that span over 1,732,603 papers with at least one mention according to Istrate et al. Further local verification resolved 1,711,333 papers with at least one mention. Additionally, of the 14,770,209 software mentions, 91112 mentions were missing a DOI, limiting the number of papers that could be linked to 14,679,097 software mentions. Of the 1,711,333 papers of the CZI dataset, 1,579,368 paper were classified by the openAlex dataset. Thus 92.3\% of the papers could be labeled. 


Additional versioning details:

The openAlex data used: version 2025aug Oct 29, 2025. zenedo: 10.5281/zenodo.17442025

The Istrate et al data used: Published Sep 19, 2022; Updated Sep 27, 2022 on Dryad. https://doi.org/10.5061/dryad.6wwpzgn2c


1. [Notebook Setup](#setup)
2. [Loading in CZI dataset](#czi_dataframe)
    1. [Loading in CZI dataset](#czi_metadata)
3. [Retrieve the clustering information from openAlex for the publications in the CZI dataset](#match)

<a id='setup'></a>

## Notebook setup
required packages and directory specifications
The notebook assumes that the dataset files are stored in a folder `data` that sits as the same level as the `notebooks` directory. The assumed directory structure is the following:

- `notebooks`
-  `data` 
    - `classification_openalex_2025`
        - clustering.tsv
    - `software_mentions`
        - `disambiguated`
            - comm_disambiguated.tsv.gz
    - `open_alex_matches`
        - matches.csv
        - not_matches.csv
        - metrics.txt

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('max_colwidth', 1000)

In [3]:
# version 2025aug Oct 29, 2025. zenedo: 10.5281/zenodo.17442025
ROOT_CLASSIFICATION_DIR = '../data/classification_openalex_2025/'

# Published Sep 19, 2022; Updated Sep 27, 2022 on Dryad. https://doi.org/10.5061/dryad.6wwpzgn2c
ROOT_DISAMBIGUATED_DIR = '../data/software_mentions/'

In [4]:

clustering_path = ROOT_CLASSIFICATION_DIR + 'clustering.tsv'

file_name = 'matches.csv' # the unique publications with the linked to openAlex metadata 
not_found_file_name = 'not_matches.csv' # the publications that did not occur in the openAlex dataset but did occur in the CZI dataset
metrics_name = "metrics.txt" # the CZI metrics
metrics_name_output = "metrics_output.txt" # the CZI metrics
matches_meta_name = "matches_meta.txt" # the matches meta data of the results
DESTINATION_FOLDER = 'open_alex_matches/'
DESTINATION_FOLDER_PATH = Path('../data/open_alex_matches/')
file_path = Path('../data/' + DESTINATION_FOLDER + file_name)
not_found_file_path = Path('../data/' + DESTINATION_FOLDER + not_found_file_name)
metrics_path = Path('../data/' + DESTINATION_FOLDER + metrics_name)
matches_meta_path = Path('../data/' + DESTINATION_FOLDER + matches_meta_name)
metrics_path_output = Path('../data/' + DESTINATION_FOLDER + metrics_name_output)




<a id='czi_dataframe'></a>

## Loading in CZI dataset
The entire CZI dataset is loaded into memory.

In [5]:
data_df = pd.read_csv(ROOT_DISAMBIGUATED_DIR + 'disambiguated/comm_disambiguated.tsv.gz', sep = '\\t', engine = 'python', compression = 'gzip')

<a id='czi_metadata'></a>
### CZI Metadata
Metrics of the CZI dataset. 

1. Number of papers with at least one mention (NaN included): 1711334
2. Number of papers with at least one mention (NaN omitted): 1711333

3. number of mentions: 14770209
4. Number of mentions with a valid doi value: 14679097
5. Number of mentions with without a valid doi value: 91112

6. Number of unique mentions: 1120109
7. Number of unique mentions (disambiguated mapping): 895209
8. Number of unique mentions (valid doi subset): 1114701
9. Number of unique mentions (disambiguated mapping) (valid doi subset): 891113
10. Percentage NaN: 0.62%

11. Wrote summary to: ..\data\open_alex_matches\metrics.txt

In [ ]:
from pathlib import Path
import pandas as pd


class DataMetrics:
    """
    Computes and reports mention and DOI metrics
    from a software mentions DataFrame.
    
    These metrics are reflected to the original paper by Istrate et al. (Software mention extraction)
    """

    def __init__(self, data_df: pd.DataFrame, metrics_path: Path) -> None:
        self.data_df = data_df.copy()
        self.metrics_path = metrics_path

    def _compute(self) -> None:
        df = self.data_df

        has_doi_mask = df["doi"].notna()
        self._data_with_doi    = df[has_doi_mask]
        self._data_without_doi = df[~has_doi_mask]

        self.papers_with_doi_nan_included = df["doi"].nunique(dropna=False)
        self.papers_with_doi_valid        = df["doi"].nunique(dropna=True)

        self.total_mentions      = len(df)
        self.mentions_with_doi   = len(self._data_with_doi)
        self.mentions_without_doi = len(self._data_without_doi)

        self.unique_mentions         = df["software"].nunique()
        self.unique_mentions_mapping = df["mapped_to_software"].nunique()

        self.unique_mentions_with_doi = self._data_with_doi["software"].nunique()
        self.unique_mentions_mapping_with_doi = self._data_with_doi["mapped_to_software"].nunique()


        self.nan_percentage = (
            self.mentions_without_doi / self.total_mentions * 100
            if self.total_mentions else 0.0
        )


    # these prashing reflect the terminology used in the original paper by Istrate et al. (Software mention extraction)
    # Specifically, Table 1: PMC OA statistics. 
    def _metric_lines(self) -> list[str]:
        return [
            f"Number of papers with at least one mention (NaN included): {self.papers_with_doi_nan_included}",
            f"Number of papers with at least one mention (NaN omitted): {self.papers_with_doi_valid}",
            "",
            f"Number of mentions: {self.total_mentions}",
            f"Number of mentions with a valid doi value: {self.mentions_with_doi}",
            f"Number of mentions without a valid doi value: {self.mentions_without_doi}",
            "",
            f"Number of unique mentions: {self.unique_mentions}",
            f"Number of unique mentions (disambiguated mapping): {self.unique_mentions_mapping}",
            f"Number of unique mentions (valid doi subset): {self.unique_mentions_with_doi}",
            f"Number of unique mentions (disambiguated mapping) (valid doi subset): {self.unique_mentions_mapping_with_doi}",
            "",
            f""
            "",
            f"Percentage NaN: {self.nan_percentage:.2f}%",
        ]

    def _execute(self) -> list[str]:
        """Recompute metrics and return the formatted lines."""
        self._compute()
        return self._metric_lines()

    def report(self) -> None:
        for line in self._execute():
            print(line)

    def report_and_save(self) -> None:
        lines = self._execute()
        for line in lines:
            print(line)
        self.metrics_path.write_text("\n".join(lines), encoding="utf-8")
        print(f"\nWrote summary to: {self.metrics_path}")

In [7]:
metrics = DataMetrics(data_df, metrics_path)
metrics.report_and_save()

Number of papers with at least one mention (NaN included): 1711334
Number of papers with at least one mention (NaN omitted): 1711333

Number of mentions: 14770209
Number of mentions with a valid doi value: 14679097
Number of mentions without a valid doi value: 91112

Number of unique mentions: 1120109
Number of unique mentions (disambiguated mapping): 97662
Number of unique mentions (valid doi subset): 1114701
Number of unique mentions (disambiguated mapping) (valid doi subset): 97598


Percentage NaN: 0.62%

Wrote summary to: ..\data\open_alex_matches\metrics.txt


<a id='match'></a>
## Retrieve the clustering information from openAlex for the publications in the CZI dataset

1. Total number of papers searched 1711333
2. Total number of matches from the Open Alex dataset 1579368
3. Total number of papers that did not match open alex dataset 131965
4. Percentage of papers that could be matched: 0.9228875969784958%

In [8]:
class DoiMatcher:
    """
    Matches software-mention DOIs against a 
    a large classification dateset in chunks by 
    post Nees Jan van Eck and Ludo Waltman

    """

    CHUNK_SIZE = 500_000

    def __init__(
        self,
        data_df: pd.DataFrame,
        clustering_path: Path,
        file_path: Path,
        not_found_file_path: Path,
        metrics_path: Path,
    ) -> None:
        self.data_df = data_df.copy()
        self.clustering_path = clustering_path
        self.file_path = file_path
        self.not_found_file_path = not_found_file_path
        self.metrics_path = metrics_path

    def _run(self) -> None:
        self.file_path.parent.mkdir(parents=True, exist_ok=True)

        # set of all software-mention DOIs to match against
        self.data_df['has_no_doi'] = self.data_df['doi'].isna()
        intermediate = self.data_df[self.data_df['has_no_doi'] == False]
        all_paper_doi_set = set(intermediate['doi'].unique())

        # accumulators
        found_dois = set()
        matched_rows_count = 0
        first_write = True

        # I chose to iterate over chunks because I can not hold the
        # entire dataframe in memory
        for chunk in pd.read_csv(
                self.clustering_path,
                sep='\t',
                engine='python',
                chunksize=self.CHUNK_SIZE,
                dtype=str
            ):
            matches = chunk['doi'].isin(all_paper_doi_set)
            found_chunk = chunk.loc[matches]

            if not found_chunk.empty:
                matched_rows_count += len(found_chunk)
                found_dois.update(found_chunk['doi'].dropna().unique())
                found_chunk.to_csv(
                    self.file_path,
                    index=False,
                    mode='w' if first_write else 'a',
                    header=first_write
                )
                first_write = False

        
        not_found_dois = sorted(all_paper_doi_set - found_dois)
        entries_not_found_df = pd.DataFrame({'doi': not_found_dois})
        entries_not_found_df.to_csv(self.not_found_file_path, index=False)

        # used for saving metrics
        self.total_doi_search    = len(all_paper_doi_set)
        self.total_doi_found     = matched_rows_count
        self.total_doi_not_found = self.total_doi_search - self.total_doi_found
        self.match_percentage    = (
            self.total_doi_found / self.total_doi_search
            if self.total_doi_search else 0.0
        )

    def _build_lines(self) -> list[str]:
        return [
            f"Total number of papers searched {self.total_doi_search}",
            f"Total number of matches from the Open Alex dataset {self.total_doi_found}",
            f"Total number of papers that did not match open alex dataset {self.total_doi_not_found}",
            f"Percentage of papers that could be matched: {self.total_doi_found / self.total_doi_search}%",
        ]

    def _execute(self) -> list[str]:
        self._run()
        return self._build_lines()

    def report_and_save(self) -> None:
        lines = self._execute()
        for line in lines:
            print(line)
        self.metrics_path.write_text("\n".join(lines), encoding="utf-8")
        print(f"\nWrote meta results to: {self.metrics_path}")

In [9]:
matcher = DoiMatcher(data_df, clustering_path, file_path, not_found_file_path, matches_meta_path)
matcher.report_and_save()

Total number of papers searched 1711333
Total number of matches from the Open Alex dataset 1579368
Total number of papers that did not match open alex dataset 131965
Percentage of papers that could be matched: 0.9228875969784958%

Wrote meta results to: ..\data\open_alex_matches\matches_meta.txt


In [10]:
matches = pd.read_csv(file_path)
not_matches = pd.read_csv(not_found_file_path)

In [11]:
# Changed files to include the compression extension
file_name = 'matches.csv.gz' # the unique publications with the linked to openAlex metadata 
not_found_file_name = 'not_matches.csv.gz' # the publications that did not occur in the openAlex dataset but did occur in the CZI dataset

file_path = Path('../data/' + DESTINATION_FOLDER + file_name)
not_found_file_path = Path('../data/' + DESTINATION_FOLDER + not_found_file_name)

matches.to_csv(file_path, compression="gzip", index=False)
not_matches.to_csv(not_found_file_path, compression="gzip", index=False)